# Lag vs. Current Driver Model: National MMG Data, Evaluated on Florida

Rebuild of `1.4-cc-lag-vs-current-food-insecurity-model.ipynb` using genuinely national
Map the Meal Gap data (all US counties/ZCTAs) instead of the Feeding Tampa Bay-scoped
export used there. Two MMG releases are combined into one panel:

- `MMG_2019-2023.xlsx` &mdash; multi-year retrospective panel, County years 2019-2023,
  ZCTA years 2020-2023 (Map the Meal Gap's ZCTA panel always starts a year later than
  the County panel).
- `MMG_2024.xlsx` &mdash; single additional year, 2024, both sheets.

Same three specifications as 1.4:

1. **Current model** &mdash; `food_insecurity_rate[t] ~ lag_food_insecurity_rate[t-1] + drivers[t]`
2. **2-year lag model** &mdash; `food_insecurity_rate[t] ~ food_insecurity_rate[t-2] + drivers[t-2]`
3. **3-year lag model** &mdash; `food_insecurity_rate[t] ~ food_insecurity_rate[t-3] + drivers[t-3]`

We train each spec on the **full national panel** (more data, more stable ridge fit) and
then evaluate the held-out year both nationally and restricted to **Florida** rows, since
predicting Florida is the actual goal -- this tests whether pooling national data helps
vs. training on Florida's own limited history alone, while the temporal holdout still
prevents any lookahead into the test year.

ALICE hardship data is Florida-only (United For ALICE publishes per state, no national
file), so it only ever contributes for Florida rows here, same as in 1.4.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

from openpyxl import load_workbook

## Configuration

In [2]:
# Resolve project paths so the notebook works from the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "external").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Two national MMG releases, combined into one panel. Both use plain "County"/"ZCTA"
# sheet names (unlike the FTB-scoped export in 1.4, which had a differently-named ZCTA sheet).
MMG_WORKBOOK_PATHS = [
    EXTERNAL_DIR / "MMG_2019-2023.xlsx",
    EXTERNAL_DIR / "MMG_2024.xlsx",
]
ZCTA_SHEET = "ZCTA"
COUNTY_SHEET = "County"
ALICE_COUNTY_PATH = EXTERNAL_DIR / "2025 ALICE - Florida Data Sheet (Lee).xlsx - County.csv"

# Evaluate against Florida specifically (the project's actual target), and also report
# the Feeding Tampa Bay sub-slice for continuity with notebook 1.4.
TARGET_STATE = "FL"
TARGET_FOOD_BANK = "Feeding Tampa Bay"

OUTPUT_PATH = PROCESSED_DIR / "lag_vs_current_model_comparison_national.csv"

MMG_WORKBOOK_PATHS, ALICE_COUNTY_PATH, OUTPUT_PATH

([WindowsPath('C:/Users/Tom/ftb-pro-bono/data/external/MMG_2019-2023.xlsx'),
  WindowsPath('C:/Users/Tom/ftb-pro-bono/data/external/MMG_2024.xlsx')],
 WindowsPath('C:/Users/Tom/ftb-pro-bono/data/external/2025 ALICE - Florida Data Sheet (Lee).xlsx - County.csv'),
 WindowsPath('C:/Users/Tom/ftb-pro-bono/data/processed/lag_vs_current_model_comparison_national.csv'))

## Helper Functions

In [3]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize spreadsheet column names for consistent downstream access."""
    out = df.copy()
    out.columns = [re.sub(r"\s+", " ", str(c).strip()) if c is not None else f"unnamed_{i}" for i, c in enumerate(out.columns)]
    return out


def read_excel_sheet_openpyxl(path: Path, sheet_name: str) -> pd.DataFrame:
    """Read a workbook sheet directly with openpyxl to avoid pandas/openpyxl version issues."""
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet_name]
    rows = ws.iter_rows(values_only=True)
    header = next(rows)

    # Drop trailing blank workbook columns so they do not become unusable dataframe fields.
    useful_cols = [i for i, value in enumerate(header) if value is not None]
    names = [header[i] for i in useful_cols]
    # Some sheets have a long tail of fully-blank rows past the real data (formatting artifacts);
    # skip any row that has no populated cells at all rather than assuming it matches the header width.
    data = [[row[i] for i in useful_cols] for row in rows if any(v is not None for v in row)]
    return clean_columns(pd.DataFrame(data, columns=names))


def read_sheet_from_workbooks(paths: list, sheet_name: str) -> pd.DataFrame:
    """Read the same-named sheet from multiple workbooks and concatenate them into one panel."""
    frames = [read_excel_sheet_openpyxl(p, sheet_name) for p in paths]
    return pd.concat(frames, ignore_index=True)


def parse_number(series: pd.Series) -> pd.Series:
    """Convert currency, comma-formatted, and percent-looking strings to numeric values."""
    cleaned = (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def parse_rate(series: pd.Series) -> pd.Series:
    """Convert rate columns to decimals, e.g. 15.2% or 15.2 becomes 0.152."""
    values = parse_number(series)
    if values.dropna().max() > 1.5:
        values = values / 100.0
    return values


def clip_rate(series: pd.Series, low: float = 0.0, high: float = 0.95) -> pd.Series:
    """Keep modeled rates inside a plausible bounded interval."""
    return series.clip(lower=low, upper=high)


def fit_weighted_ridge(X: pd.DataFrame, y: pd.Series, weights: pd.Series, alpha: float = 1.0) -> dict:
    """Fit a population-weighted ridge regression using only NumPy."""
    # Standardize features so the ridge penalty treats each predictor comparably.
    means = X.mean()
    stds = X.std(ddof=0).replace(0, 1)
    X_scaled = (X - means) / stds

    # Add an intercept column and apply square-root weights for weighted least squares.
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled.to_numpy(dtype=float)])
    y_values = y.to_numpy(dtype=float)
    w = np.sqrt(np.maximum(weights.fillna(weights.median()).to_numpy(dtype=float), 1.0))
    Xw = X_design * w[:, None]
    yw = y_values * w

    # Penalize feature coefficients but not the intercept.
    penalty = np.eye(X_design.shape[1]) * alpha
    penalty[0, 0] = 0
    coef = np.linalg.solve(Xw.T @ Xw + penalty, Xw.T @ yw)
    return {"features": list(X.columns), "means": means, "stds": stds, "coef": coef}


def predict_weighted_ridge(model: dict, X: pd.DataFrame) -> pd.Series:
    """Generate predictions from the fitted weighted ridge model."""
    X_scaled = (X[model["features"]] - model["means"]) / model["stds"]
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled.to_numpy(dtype=float)])
    return pd.Series(X_design @ model["coef"], index=X.index)


def weighted_metrics(frame: pd.DataFrame, actual_col: str, pred_col: str, weight_col: str = "population") -> dict:
    """Population-weighted MAE / RMSE / mean error."""
    if frame.empty:
        return {"weighted_mae": np.nan, "weighted_rmse": np.nan, "weighted_mean_error": np.nan, "n": 0}
    w = frame[weight_col].fillna(1).clip(lower=1)
    err = frame[pred_col] - frame[actual_col]
    return {
        "weighted_mae": float(np.average(err.abs(), weights=w)),
        "weighted_rmse": float(np.sqrt(np.average(err ** 2, weights=w))),
        "weighted_mean_error": float(np.average(err, weights=w)),
        "n": int(len(frame)),
    }

## Load Source Data

In [4]:
# Load and combine both national MMG releases, plus Florida ALICE hardship data.
zcta_raw = read_sheet_from_workbooks(MMG_WORKBOOK_PATHS, ZCTA_SHEET)
county_raw = read_sheet_from_workbooks(MMG_WORKBOOK_PATHS, COUNTY_SHEET)
alice_county_raw = clean_columns(pd.read_csv(ALICE_COUNTY_PATH, dtype=str))

len(zcta_raw), len(county_raw), len(alice_county_raw)

(232864, 19094, 603)

## Build the ZIP-County-Year Panel

Same feature construction as notebook 1.3/1.4, with two changes for the national data:

1. Both files' County sheets carry a real `Year` column, so years are read directly
   instead of assumed positionally (1.4's County sheet had no Year column and guessed
   2019-2023 by row order within each county -- that assumption is gone here).
2. `state_fips` is derived from the first two digits of `county_fips` rather than read
   from a `State FIPS` column, since the 2024 release's ZCTA sheet doesn't include that
   column at all.

In [5]:
zcta = zcta_raw.copy()
county = county_raw.copy()
alice_county = alice_county_raw.copy()

# -----------------------------
# County-level feature table
# -----------------------------
county["county_fips"] = county["FIPS"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
county["year"] = pd.to_numeric(county["Year"], errors="coerce").astype("Int64")
county["state_fips"] = county["county_fips"].str[:2]
county["county_population"] = parse_number(county["Total Population (5 Year ACS)"])
county["county_child_population"] = parse_number(county["Total Child Population (5 Year ACS)"])
county["county_child_population_share"] = county["county_child_population"] / county["county_population"]
county["county_percent_white_non_hispanic"] = parse_rate(county["Percent White, non-Hispanic (5 Year ACS)"])
county["county_cost_per_meal"] = parse_number(county["Cost Per Meal"])
county["county_weighted_food_cost_index"] = parse_number(county["Weighted Index"])
county["county_snap_threshold"] = parse_number(county["SNAP Threshold"])
county["county_rural_urban_code_2023"] = parse_number(county["Rural-Urban Continuum Code (2023)"])

county_features = county[[
    "county_fips", "year", "county_child_population_share", "county_percent_white_non_hispanic",
    "county_cost_per_meal", "county_weighted_food_cost_index", "county_snap_threshold",
    "county_rural_urban_code_2023"
]].dropna(subset=["county_fips", "year"]).copy()
county_features["year"] = county_features["year"].astype(int)
# Two source files should never share a county-year; drop-duplicates is a safety net, not the primary guard.
county_features = county_features.drop_duplicates(subset=["county_fips", "year"])

# -----------------------------
# Florida ALICE county features
# -----------------------------
alice_county["year"] = pd.to_numeric(alice_county["Year"], errors="coerce").astype("Int64")
alice_county["county_fips"] = alice_county["GEO id2"].astype(str).str.zfill(5)
alice_county["alice_households"] = parse_number(alice_county["ALICE Households"])
alice_county["alice_poverty_households"] = parse_number(alice_county["Poverty Households"])
alice_county["alice_total_households"] = parse_number(alice_county["Households"])
alice_county["alice_financial_insecurity_rate"] = (
    alice_county["alice_households"] + alice_county["alice_poverty_households"]
) / alice_county["alice_total_households"]
alice_county["alice_poverty_household_rate"] = alice_county["alice_poverty_households"] / alice_county["alice_total_households"]
alice_county["alice_threshold_under_65"] = parse_number(alice_county["ALICE Threshold - HH under 65"])
alice_county["alice_threshold_65_plus"] = parse_number(alice_county["ALICE Threshold - HH 65 years and over"])

alice_features = alice_county[[
    "county_fips", "year", "alice_financial_insecurity_rate", "alice_poverty_household_rate",
    "alice_threshold_under_65", "alice_threshold_65_plus"
]].dropna(subset=["county_fips", "year"]).copy()
alice_features["year"] = alice_features["year"].astype(int)

# -----------------------------
# ZCTA panel table
# -----------------------------
zcta["year"] = pd.to_numeric(zcta["Year"], errors="coerce").astype("Int64")
zcta["county_fips"] = zcta["County FIPS"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
zcta["state_fips"] = zcta["county_fips"].str[:2]
zcta["zcta"] = zcta["ZCTA"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
zcta["row_id"] = zcta["state_fips"] + "_" + zcta["county_fips"] + "_" + zcta["zcta"]

zcta["population"] = parse_number(zcta["Total Population (5 Year ACS)"])
zcta["food_insecurity_rate"] = parse_rate(zcta["Overall Food Insecurity Rate"])
zcta["food_insecure_persons"] = parse_number(zcta["# of Food Insecure Persons Overall"])
zcta["unemployment_rate"] = parse_rate(zcta["Unemployment Rate (1 Yr BLS)"])
zcta["poverty_rate"] = parse_rate(zcta["Poverty Rate (5 Yr ACS)"])
zcta["percent_black"] = parse_rate(zcta["Percent Black (5 Yr ACS)"])
zcta["percent_hispanic"] = parse_rate(zcta["Percent Hispanic (any race) (5 Year ACS)"])
zcta["median_income"] = parse_number(zcta["Median Income (5 Yr ACS)"])
zcta["log_median_income"] = np.log(zcta["median_income"].replace(0, np.nan))
zcta["homeownership_rate"] = parse_rate(zcta["Homeownership Rate (5 Yr ACS)"])
zcta["disability_rate"] = parse_rate(zcta["Disability Rate (5 Yr ACS)"])

model_cols = [
    "row_id", "state_fips", "county_fips", "zcta", "Geography", "County, State", "State",
    "Food Bank 1 ID", "Food Bank 1", "Food Bank 2 ID", "Food Bank 2", "year", "population",
    "food_insecurity_rate", "food_insecure_persons", "unemployment_rate", "poverty_rate",
    "percent_black", "percent_hispanic", "median_income", "log_median_income",
    "homeownership_rate", "disability_rate"
]
panel = zcta[model_cols].dropna(subset=["year", "row_id"]).copy()
panel["year"] = panel["year"].astype(int)
panel = panel.drop_duplicates(subset=["row_id", "year"])

panel = panel.merge(county_features, on=["county_fips", "year"], how="left")
panel = panel.merge(alice_features, on=["county_fips", "year"], how="left")
panel["has_alice_data"] = panel["alice_financial_insecurity_rate"].notna().astype(int)
panel = panel.sort_values(["row_id", "year"]).reset_index(drop=True)

# Fill driver gaps with state medians first, then national medians as a final fallback.
numeric_features = [
    "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic", "log_median_income",
    "homeownership_rate", "disability_rate", "county_child_population_share",
    "county_percent_white_non_hispanic", "county_cost_per_meal", "county_weighted_food_cost_index",
    "county_snap_threshold", "county_rural_urban_code_2023", "alice_financial_insecurity_rate",
    "alice_poverty_household_rate", "alice_threshold_under_65", "alice_threshold_65_plus", "has_alice_data"
]
for col in numeric_features:
    panel[col] = panel.groupby("State")[col].transform(lambda s: s.fillna(s.median()))
    panel[col] = panel[col].fillna(panel[col].median())

observed_years = sorted(panel["year"].dropna().unique().tolist())
print(f"Observed years: {observed_years}")
print(f"Rows: {len(panel):,}, unique row_ids: {panel['row_id'].nunique():,}")
print(f"States: {panel['State'].nunique()}, Florida rows: {(panel['State'] == TARGET_STATE).sum():,}")
display(panel.head())

Observed years: [2020, 2021, 2022, 2023, 2024]
Rows: 232,268, unique row_ids: 46,947
States: 52, Florida rows: 5,837


,row_id,state_fips,county_fips,zcta,Geography,"County, State",State,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,...,county_percent_white_non_hispanic,county_cost_per_meal,county_weighted_food_cost_index,county_snap_threshold,county_rural_urban_code_2023,alice_financial_insecurity_rate,alice_poverty_household_rate,alice_threshold_under_65,alice_threshold_65_plus,has_alice_data
0,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54,"Montgomery Area Food Bank, Inc.",NaN,...,0.740,3.22,0.99,1.3,3.0,0.462131,0.130891,61980.0,58284.0,0
1,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54,Montgomery Area Food Bank,NaN,...,0.731,3.60,1.00,1.3,3.0,0.462131,0.130891,61980.0,58284.0,0
2,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54,Heart of Alabama Food Bank,NaN,...,0.726,4.01,1.01,1.3,2.0,0.462131,0.130891,61980.0,58284.0,0
3,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54,Heart of Alabama Food Bank,NaN,...,0.717,3.64,1.02,1.3,2.0,0.462131,0.130891,61980.0,58284.0,0
4,01_01001_36003,01,01001,36003,ZCTA5 36003,AUTAUGA,AL,54,Heart of Alabama Food Bank,NaN,...,0.706,3.68,0.99,1.3,2.0,0.462131,0.130891,61980.0,58284.0,0


## Driver Columns and Lag Specifications

In [6]:
# Same 18 drivers used in notebooks 1.3/1.4, minus the lagged rate itself (added separately per spec).
driver_cols = [
    "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic",
    "log_median_income", "homeownership_rate", "disability_rate",
    "county_child_population_share", "county_percent_white_non_hispanic", "county_cost_per_meal",
    "county_weighted_food_cost_index", "county_snap_threshold", "county_rural_urban_code_2023",
    "alice_financial_insecurity_rate", "alice_poverty_household_rate", "alice_threshold_under_65",
    "alice_threshold_65_plus", "has_alice_data",
]

LAG_SPECS = {
    "current (X[t], rate[t-1])": (0, 1),
    "2-year lag (X[t-2], rate[t-2])": (2, 2),
    "3-year lag (X[t-3], rate[t-3])": (3, 3),
}
LAG_SPECS

{'current (X[t], rate[t-1])': (0, 1),
 '2-year lag (X[t-2], rate[t-2])': (2, 2),
 '3-year lag (X[t-3], rate[t-3])': (3, 3)}

## Build a Lagged Transition Table

In [7]:
def build_transition_table(panel: pd.DataFrame, driver_lag: int, rate_lag: int, driver_cols: list) -> pd.DataFrame:
    """Pair each row's current-year target with drivers/rate measured `driver_lag`/`rate_lag` years earlier."""
    df = panel.sort_values(["row_id", "year"]).copy()
    g = df.groupby("row_id")

    out = df[["row_id", "year", "population", "food_insecurity_rate", "Food Bank 1", "State"]].copy()
    out["lag_food_insecurity_rate"] = g["food_insecurity_rate"].shift(rate_lag)
    for col in driver_cols:
        out[col] = g[col].shift(driver_lag)
    return out

## Fit, Tune, and Evaluate a Lag Specification

In [8]:
def evaluate_lag_spec(
    panel: pd.DataFrame,
    label: str,
    driver_lag: int,
    rate_lag: int,
    driver_cols: list,
    target_state: str,
    target_food_bank: str,
    alpha_grid=(0.01, 0.1, 0.5, 1.0, 2.0),
    n_folds: int = 5,
    random_state: int = 42,
) -> dict:
    """Fit a population-weighted ridge model on the full national panel for one lag specification.

    Trains on all states (more data, more stable ridge fit) but reports metrics nationally,
    for Florida overall, and for the Feeding Tampa Bay slice specifically -- Florida is the
    metric that matters here, FTB is kept for continuity with notebook 1.4.

    Uses a time-based holdout (train on earlier transition years, evaluate on the latest
    available one) whenever at least two transition years exist for this lag. Falls back to
    5-fold cross-validation across ZIP-county rows when the lag leaves only one usable
    transition year.
    """
    feature_cols = ["lag_food_insecurity_rate"] + driver_cols
    table = build_transition_table(panel, driver_lag, rate_lag, driver_cols)
    valid = table.dropna(subset=["food_insecurity_rate", "lag_food_insecurity_rate", "population"] + driver_cols).copy()

    valid["is_fl"] = valid["State"].astype(str).str.upper() == target_state.upper() if target_state else True
    valid["is_ftb"] = (
        valid["Food Bank 1"].astype(str).str.contains(target_food_bank, case=False, na=False)
        if target_food_bank else True
    )

    available_years = sorted(valid["year"].unique().tolist())
    if len(available_years) == 0:
        raise ValueError(f"{label}: no usable transitions for driver_lag={driver_lag}, rate_lag={rate_lag}.")

    def score(train_df, test_df, alpha):
        m = fit_weighted_ridge(train_df[feature_cols], train_df["food_insecurity_rate"], train_df["population"], alpha=alpha)
        pred = clip_rate(predict_weighted_ridge(m, test_df[feature_cols]))
        return test_df.assign(prediction=pred), m

    if len(available_years) >= 2:
        method = "time_holdout"
        latest_year = available_years[-1]
        train_full = valid.loc[valid["year"] < latest_year].copy()
        test = valid.loc[valid["year"] == latest_year].copy()

        alpha_rows = []
        for alpha in alpha_grid:
            scored, _ = score(train_full, test, alpha)
            fl = weighted_metrics(scored.loc[scored["is_fl"]], "food_insecurity_rate", "prediction")
            nat = weighted_metrics(scored, "food_insecurity_rate", "prediction")
            alpha_rows.append({"alpha": alpha, "fl_weighted_mae": fl["weighted_mae"], "national_weighted_mae": nat["weighted_mae"]})
        alpha_df = pd.DataFrame(alpha_rows)
        selection_metric = "fl_weighted_mae" if alpha_df["fl_weighted_mae"].notna().any() else "national_weighted_mae"
        selected_alpha = float(alpha_df.sort_values([selection_metric, "national_weighted_mae", "alpha"]).iloc[0]["alpha"])

        scored, final_model = score(train_full, test, selected_alpha)
        train_years = sorted(train_full["year"].unique().tolist())
        n_train = len(train_full)
        n_test = len(test)

    else:
        method = "5_fold_cv_fallback"
        shuffled = valid.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
        shuffled["fold"] = np.arange(len(shuffled)) % n_folds

        def run_cv(alpha):
            fold_frames = []
            for f in range(n_folds):
                train_fold = shuffled.loc[shuffled["fold"] != f]
                test_fold = shuffled.loc[shuffled["fold"] == f]
                scored_fold, _ = score(train_fold, test_fold, alpha)
                fold_frames.append(scored_fold)
            return pd.concat(fold_frames, ignore_index=True)

        alpha_rows = []
        for alpha in alpha_grid:
            cv_scored = run_cv(alpha)
            fl = weighted_metrics(cv_scored.loc[cv_scored["is_fl"]], "food_insecurity_rate", "prediction")
            nat = weighted_metrics(cv_scored, "food_insecurity_rate", "prediction")
            alpha_rows.append({"alpha": alpha, "fl_weighted_mae": fl["weighted_mae"], "national_weighted_mae": nat["weighted_mae"]})
        alpha_df = pd.DataFrame(alpha_rows)
        selection_metric = "fl_weighted_mae" if alpha_df["fl_weighted_mae"].notna().any() else "national_weighted_mae"
        selected_alpha = float(alpha_df.sort_values([selection_metric, "national_weighted_mae", "alpha"]).iloc[0]["alpha"])

        scored = run_cv(selected_alpha)
        # Final model fit on all available rows, for coefficient inspection only -- not what generated the metrics above.
        final_model = fit_weighted_ridge(shuffled[feature_cols], shuffled["food_insecurity_rate"], shuffled["population"], alpha=selected_alpha)
        train_years = available_years
        n_train = len(shuffled)
        n_test = len(scored)

    national_metrics = weighted_metrics(scored, "food_insecurity_rate", "prediction")
    fl_metrics = weighted_metrics(scored.loc[scored["is_fl"]], "food_insecurity_rate", "prediction")
    ftb_metrics = weighted_metrics(scored.loc[scored["is_ftb"]], "food_insecurity_rate", "prediction")

    return {
        "label": label,
        "driver_lag": driver_lag,
        "rate_lag": rate_lag,
        "validation_method": method,
        "selected_alpha": selected_alpha,
        "available_years": available_years,
        "train_years": train_years,
        "n_train": n_train,
        "n_test": n_test,
        "national_weighted_mae": national_metrics["weighted_mae"],
        "national_weighted_rmse": national_metrics["weighted_rmse"],
        "national_weighted_mean_error": national_metrics["weighted_mean_error"],
        "national_n": national_metrics["n"],
        "fl_weighted_mae": fl_metrics["weighted_mae"],
        "fl_weighted_rmse": fl_metrics["weighted_rmse"],
        "fl_weighted_mean_error": fl_metrics["weighted_mean_error"],
        "fl_n": fl_metrics["n"],
        "ftb_weighted_mae": ftb_metrics["weighted_mae"],
        "ftb_weighted_rmse": ftb_metrics["weighted_rmse"],
        "ftb_weighted_mean_error": ftb_metrics["weighted_mean_error"],
        "ftb_n": ftb_metrics["n"],
        "model": final_model,
        "feature_cols": feature_cols,
    }

## Run All Three Specifications

If a specification prints `validation_method: 5_fold_cv_fallback`, there wasn't enough
time-series depth for a genuine out-of-time holdout at that lag -- treat its metrics as
lower-confidence than the ones using `time_holdout`.

In [9]:
results = {}
for label, (driver_lag, rate_lag) in LAG_SPECS.items():
    result = evaluate_lag_spec(
        panel=panel,
        label=label,
        driver_lag=driver_lag,
        rate_lag=rate_lag,
        driver_cols=driver_cols,
        target_state=TARGET_STATE,
        target_food_bank=TARGET_FOOD_BANK,
    )
    results[label] = result
    print(
        f"{label}: method={result['validation_method']}, alpha={result['selected_alpha']:g}, "
        f"train_years={result['train_years']}, n_train={result['n_train']:,}, n_test={result['n_test']:,}"
    )

current (X[t], rate[t-1]): method=time_holdout, alpha=2, train_years=[2021, 2022, 2023], n_train=109,148, n_test=36,591


2-year lag (X[t-2], rate[t-2]): method=time_holdout, alpha=2, train_years=[2022, 2023], n_train=72,217, n_test=36,369


3-year lag (X[t-3], rate[t-3]): method=time_holdout, alpha=2, train_years=[2023], n_train=35,922, n_test=36,035


In [10]:
comparison_df = pd.DataFrame([
    {
        "specification": r["label"],
        "validation_method": r["validation_method"],
        "selected_alpha": r["selected_alpha"],
        "train_years": r["train_years"],
        "n_train": r["n_train"],
        "n_test": r["n_test"],
        "national_weighted_mae": r["national_weighted_mae"],
        "national_weighted_rmse": r["national_weighted_rmse"],
        "national_weighted_mean_error": r["national_weighted_mean_error"],
        "national_n": r["national_n"],
        "fl_weighted_mae": r["fl_weighted_mae"],
        "fl_weighted_rmse": r["fl_weighted_rmse"],
        "fl_weighted_mean_error": r["fl_weighted_mean_error"],
        "fl_n": r["fl_n"],
        "ftb_weighted_mae": r["ftb_weighted_mae"],
        "ftb_weighted_rmse": r["ftb_weighted_rmse"],
        "ftb_weighted_mean_error": r["ftb_weighted_mean_error"],
        "ftb_n": r["ftb_n"],
    }
    for r in results.values()
])
display(comparison_df)
comparison_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved comparison table to: {OUTPUT_PATH}")

,specification,validation_method,selected_alpha,train_years,n_train,n_test,national_weighted_mae,national_weighted_rmse,national_weighted_mean_error,national_n,fl_weighted_mae,fl_weighted_rmse,fl_weighted_mean_error,fl_n,ftb_weighted_mae,ftb_weighted_rmse,ftb_weighted_mean_error,ftb_n
0,"current (X[t], rate[t-1])",time_holdout,2.0,"[2021, 2022, 2023]",109148,36591,0.015114,0.020348,0.001565,36591,0.015985,0.021628,0.001413,1055,0.010772,0.013942,0.004146,233
1,"2-year lag (X[t-2], rate[t-2])",time_holdout,2.0,"[2022, 2023]",72217,36369,0.027125,0.032014,0.022556,36369,0.027610,0.031310,0.023903,1055,0.026498,0.029679,0.025450,233
2,"3-year lag (X[t-3], rate[t-3])",time_holdout,2.0,[2023],35922,36035,0.020726,0.028162,-0.014950,36035,0.017870,0.024768,-0.010949,1060,0.014811,0.018913,-0.009094,236


Saved comparison table to: C:\Users\Tom\ftb-pro-bono\data\processed\lag_vs_current_model_comparison_national.csv


## Compare Coefficients Across Specifications

Standardized ridge coefficients (each predictor scaled to mean 0 / std 1 before fitting),
fit on the full national training set for each spec.

In [11]:
coef_frame = pd.DataFrame({"feature": ["intercept"] + results[list(LAG_SPECS)[0]]["feature_cols"]})
for label, r in results.items():
    coef_frame[label] = r["model"]["coef"]
display(coef_frame)

,feature,"current (X[t], rate[t-1])","2-year lag (X[t-2], rate[t-2])","3-year lag (X[t-3], rate[t-3])"
0,intercept,0.132605,0.145220,0.148762
1,lag_food_insecurity_rate,0.044930,0.032586,0.033882
2,unemployment_rate,0.002788,0.001220,-0.000839
3,poverty_rate,0.007715,0.005151,0.002923
4,percent_black,-0.000702,-0.000659,0.001428
5,percent_hispanic,0.001435,0.003459,0.004670
6,log_median_income,0.005353,-0.004861,-0.004247
7,homeownership_rate,-0.002296,-0.003091,-0.004426
8,disability_rate,0.004236,0.003549,0.003262
9,county_child_population_share,-0.000812,-0.000882,-0.001067


## Notes and Limitations

- Trained on the full national panel (all states), evaluated nationally, on Florida, and
  on the Feeding Tampa Bay slice specifically -- Florida is the metric that matters for
  this project, not national.
- ALICE hardship features are Florida-only; every non-Florida row falls back to the
  state/national median fill, same mechanism as missing data elsewhere.
- Two source-file schema differences were handled here: the 2024 release's ZCTA sheet
  has no `State FIPS` column (state is derived from `county_fips` instead), and its County
  sheet has an extra `NIQ Data Imputed` column we don't use. Neither file's County sheet
  needed the positional year-guessing that 1.4 required, since both have a real `Year` column.
- As before, a `5_fold_cv_fallback` result means there wasn't enough time-series depth for
  a genuine holdout at that lag -- treat it as a rough signal, not a robust validation.
- This notebook compares model specifications; it does not produce a recursive forecast.